In [5]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [6]:
# Create an API clients
from anthropic import Anthropic

client = Anthropic()
# NOTE: the Claude 5 family (claude-sonnet-5, claude-opus-4-8) removed the
# `temperature` parameter — it returns a 400. This notebook demonstrates
# temperature, so it pins to Sonnet 4.6, which still accepts it.
model = "claude-sonnet-4-6"

In [7]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def chat(messages, system=None, temperature=1.0):
    # anthropic 1.x removed `temperature` from messages.create() — passing it directly
    # raises TypeError. It is still accepted by the API on pre-5 models, so send it
    # through extra_body.
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return next(block.text for block in message.content if block.type == "text")

In [14]:
messages = []

add_user_message(messages,
    "Generate a one sentence movie idea"
)
answer = chat(messages, temperature=1.0)

answer

"Here's a movie idea:\n\n**A burned-out air traffic controller begins receiving transmissions from a plane that disappeared 30 years ago, and must guide it safely home before the passengers discover they've been gone at all.**"

## Why `temperature` was removed on newer models — and how to manage variety instead

This notebook pins `model` to Sonnet 4.6 because it demonstrates `temperature`. On the Claude 5 family (`claude-sonnet-5`, `claude-opus-4-8`) and Fable 5, `temperature` (along with `top_p` / `top_k`) was **removed** and returns a 400.

**Two separate removals — don't conflate them:**

- **Model level** — the Claude 5 family rejects `temperature` with a **400**.
- **SDK level** — `anthropic` **1.x** dropped `temperature` from `messages.create()`
  entirely; passing it raises `TypeError` before any request is sent. The parameter is
  still valid at the API level for pre-5 models, so the `chat()` helper above routes it
  through **`extra_body`**.

**Why it was removed:**

- Once **adaptive thinking** became the default, the model itself owns the explore/exploit tradeoff that people reached for `temperature` to control. A per-request sampling knob was fighting the reasoning process rather than complementing it.
- `temperature=0` never actually guaranteed deterministic output — there's nondeterminism in the serving stack regardless, so repeated calls could still differ. Removing it kills a false expectation.

**The alternative — there's no drop-in replacement; the single knob split into two:**

| What you wanted `temperature` for | Now do this |
|---|---|
| **Variety / creativity** (high temperature) | Steer with the **prompt** — ask for surprising, varied, off-distribution output. |
| **Focus / determinism** (low temperature) | Lower `output_config={"effort": ...}` (`low`→`max`) plus a tightly-scoped prompt. |

Note that `effort` controls **reasoning depth and token spend**, not sampling randomness — it is *not* a rename of `temperature`. Creativity is now a prompting concern; `effort` is a cost/thoroughness concern. The cell below shows the variety case: a prompt-driven "off-the-wall" request on `claude-sonnet-5`, which doesn't accept `temperature` at all.

In [ ]:
# Managing output variety WITHOUT temperature, on a current model.
#
# There's no drop-in replacement for `temperature`. The single knob split in two:
#   - variety / creativity (what HIGH temperature was for) -> steer with the PROMPT
#   - focus / determinism  (what LOW  temperature was for) -> lower `effort` + a tight prompt
# `output_config={"effort": ...}` controls reasoning depth and token spend, not
# sampling randomness — so prompting is how you now ask for an off-beat answer.

messages = []
add_user_message(
    messages,
    "Generate a one sentence movie idea. "
    "Make it surprising and off-the-wall — avoid the obvious, generic premise."
)

answer = next(
    block.text
    for block in client.messages.create(
        model="claude-sonnet-5",             # temperature would 400 on this model
        max_tokens=1000,
        messages=messages,
        # output_config={"effort": "low"},   # <- dial effort DOWN for tighter, more focused output
    ).content
    if block.type == "text"
)

answer